# 06.9 — Per-batch prior correction

The confidence loss ([06.6](06.6_confidence_routing.ipynb), [06.7](06.7_the_confidence_pd_controller.ipynb)) doesn't score the raw batch confidence — it scores a *dataset-level* running estimate blended from history and the current batch: `updated = history·(1−γ) + batch·γ`, where `γ = batch_fraction`. That blend has a quiet side effect: because the current batch contributes only a fraction `γ` of `updated`, the gradient flowing back to the confidence head is *scaled by γ*. So a hyperparameter (batch size) secretly rescales the learning signal. The `want_batch_correction` flag (Eq. 10) divides the loss by `γ` to cancel it. This notebook is that one subtle correction.

This notebook covers:

* Why the confidence loss uses a dataset blend, not the raw batch.
* How the blend's `γ` factor leaks into the gradient (the chain rule).
* The `1/γ` correction that makes the gradient batch-size-independent.
* The `want_batch_correction` flag and the stop-gradient it depends on.

**Prerequisites:** [06.6 confidence routing](06.6_confidence_routing.ipynb), [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb).

## Section 1 — What MATLAB does

`cgg_lossConfidence.m`'s local `compute_confidence_branch` (lines 86-162) blends a running "dataset confidence" and optionally corrects the gradient:

```matlab
% Dataset-confidence blend (Eq. 7): current batch contributes fraction gamma
updated = History .* (1 - gamma) + mean(batch) .* gamma;   % History is stop-gradient
loss    = distance(updated, 1);                            % push confidence up

if WantDatasetConfidence && WantBatchCorrection
    loss = loss ./ gamma;      % Eq. 10: explicit gradient correction
end
```

`gamma` (`BatchFraction`) is the fraction of the whole dataset in this minibatch. The correction — dividing by `gamma` — exists purely to undo the gradient attenuation the blend introduces. The port reproduces both the blend and the optional correction (Critical Note #29, subtleties #4 and #5).

## Section 2 — The Python concepts you need

### 2.1 — Why a dataset blend, not the raw batch

A single minibatch is a noisy sample of the model's confidence. The confidence loss wants to regularize the model's confidence *over the whole dataset*, so it maintains a running estimate — `updated = history·(1−γ) + batch·γ` — that mixes the accumulated history with the current batch, weighted by how much of the dataset this batch represents (`γ = batch_fraction`). This is the same running-state idea as the EMA loss priors ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)) and the confidence history ([06.6](06.6_confidence_routing.ipynb)); here it's tuned by the batch fraction. Critically, the *history* term is **stop-gradient** (detached, [02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)) — only the `batch·γ` part carries gradient.

### 2.2 — The blend leaks γ into the gradient

Because only `batch·γ` carries gradient, the derivative of `updated` with respect to the confidence head's output is exactly `γ`. So *every* gradient flowing back through the confidence loss is multiplied by `γ`. Watch it — the gradient shrinks in lockstep with the batch fraction:

In [ ]:
import torch
import torch.nn.functional as F

def confidence_grad(gamma, correct):
    head = torch.tensor([0.6], requires_grad=True)   # the confidence head's output
    batch_mean = head.mean()
    history = torch.tensor([0.8]).detach()           # stop-gradient (subtlety #4)
    updated = history * (1 - gamma) + batch_mean * gamma   # the dataset blend
    loss = F.l1_loss(updated, torch.ones_like(updated))    # push toward confidence 1
    if correct:
        loss = loss / gamma                          # Eq. 10 correction
    loss.backward()
    return head.grad.item()

print("WITHOUT correction — gradient scales with the batch fraction γ:")
for g in (1.0, 0.5, 0.1):
    print(f"  γ={g}: d(loss)/d(head) = {confidence_grad(g, correct=False):+.4f}")

The gradient is `−1.0, −0.5, −0.1` for `γ = 1.0, 0.5, 0.1` — exactly proportional to `γ`. That means if you halve your batch size (halving `γ`), you silently halve the confidence loss's push, even though you changed *nothing* about the confidence objective. A hyperparameter that should only affect *estimation smoothing* has leaked into the *gradient magnitude*. That's the bug the correction fixes.

### 2.3 — The `1/γ` correction cancels it

In [ ]:
print("WITH correction (loss / γ) — gradient is now γ-independent:")
for g in (1.0, 0.5, 0.1):
    print(f"  γ={g}: d(loss)/d(head) = {confidence_grad(g, correct=True):+.4f}")
print("→ constant −1.0: dividing the loss by γ cancels the γ the blend introduced.")

Dividing the loss by `γ` multiplies its gradient by `1/γ`, and `(1/γ)·γ = 1`. The `γ` from the blend and the `1/γ` from the correction annihilate, leaving a gradient that depends only on the confidence objective — not on the batch fraction. Now you can change batch size freely and the confidence loss behaves identically. This is "Eq. 10" in the MATLAB derivation: an *explicit gradient correction*.

### 2.4 — Why the stop-gradient matters

The correction is exactly `1/γ` *only because* the history term is detached (§2.1). If history carried gradient, `∂updated/∂head` would be more complicated (the history itself would depend on past head outputs), and a clean `1/γ` wouldn't cancel it. The stop-gradient (subtlety #4) is what makes the blend's gradient a clean `γ`, which is what makes the `1/γ` correction exact. The two subtleties are a matched pair.

### 2.5 — When it's on

`want_batch_correction` is a flag — off by default in the routing kernel, on when the config requests the batch-fraction-independent gradient. It only does anything when `want_dataset_confidence` is also on (no blend → no `γ` to correct). It's an ablation knob and a numerical-consistency tool: turn it on when you want the confidence loss to be invariant to batch size.

## Section 3 — The `neural_data_decoding` implementation

`_compute_confidence_stream_loss` — the blend, the loss, and the correction:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/confidence.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "batch_mean = batch_instances.mean()" in line)
end = next(j for j in range(i, len(src)) if "return stream_loss" in src[j])
for k in range(i, end + 1):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `hist_detached = historical.detach()` — the stop-gradient (§2.4, subtlety #4). Only `batch_mean * batch_fraction` carries gradient.
* `updated = hist_detached * (1 - batch_fraction) + batch_mean * batch_fraction` — the dataset blend (Eq. 7, subtlety #5). When `want_dataset_confidence` is off, it collapses to the raw `batch_mean`.
* The loss pushes `updated` toward `target = 1.0` (maximum confidence) — the confidence *budget* regularizer. The [06.7](06.7_the_confidence_pd_controller.ipynb) P-controller's `beta` scales this push so the *net* equilibrium is mean confidence 0.5, not 1.0. Loss-pushes-up, controller-throttles: the two close the loop.
* `stream_loss = raw_loss / batch_fraction` under `want_batch_correction` — Eq. 10, the `1/γ` (§2.3). Guarded by `want_dataset_confidence` (no blend, nothing to correct).
* `return ..., updated.detach()` — the new value is detached before becoming next batch's history, so the running state never accumulates a gradient graph ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb) threads it forward).

## Section 4 — Hands-on exercises

### Exercise 4.1 — the correction is exactly `1/γ`

For several `γ`, show that the corrected gradient equals the uncorrected gradient times `1/γ`.

In [ ]:
# Reveal:
for g in (0.5, 0.25, 0.1):
    raw = confidence_grad(g, correct=False)
    cor = confidence_grad(g, correct=True)
    print(f"  γ={g}: uncorrected {raw:+.4f} × (1/γ={1/g:.1f}) = {raw/g:+.4f}  vs corrected {cor:+.4f}")
print("→ the correction is precisely a 1/γ rescale of the gradient.")

### Exercise 4.2 — no blend, no correction

Show that with `want_dataset_confidence` off (raw batch mean, no blend), `γ` doesn't enter the gradient at all — so the correction would be meaningless.

In [ ]:
# Reveal:
def grad_no_blend(gamma):
    head = torch.tensor([0.6], requires_grad=True)
    updated = head.mean()                 # raw batch mean — NO blend, γ absent
    loss = F.l1_loss(updated, torch.ones_like(updated))
    loss.backward()
    return head.grad.item()

for g in (1.0, 0.5, 0.1):
    print(f"  γ={g}: gradient (no blend) = {grad_no_blend(g):+.4f}")
print("→ identical regardless of γ: without the blend there's no γ factor, so no correction is needed.")

## Section 5 — Common errors

### Confidence loss weakens when you shrink the batch

The classic symptom of the un-corrected blend (§2.2): smaller `batch_fraction` → smaller confidence gradient. Enable `want_batch_correction` to make the confidence objective batch-size-independent.

### Enabling batch correction without dataset confidence

`want_batch_correction` only acts when `want_dataset_confidence` is on (§2.5) — with no blend there's no `γ` in the gradient to cancel. The flag is a no-op otherwise; if you expected an effect, the blend isn't active.

### History not detached → the `1/γ` is no longer exact

The correction assumes the stop-gradient (§2.4). If you forget to detach the history, `∂updated/∂head` is no longer a clean `γ`, and dividing by `γ` over- or under-corrects. Detach the history (subtlety #4).

### Double-correcting

Divide by `γ` once. Applying the `1/γ` twice (e.g. also scaling the confidence weight by `1/γ` elsewhere) over-amplifies the confidence loss and can destabilize the multi-objective sum ([06.1](06.1_multi_task_losses_overview.ipynb)).

### Confusing batch_fraction with a learning rate

`γ` is the fraction of the *dataset* in the batch (an estimation-smoothing knob), not a step size. It governs how fast the dataset-confidence estimate tracks new batches — like `prior_proportion` for the loss priors ([06.4 §2.2](06.4_the_ema_prior_normalization_deep_dive.ipynb)), not like `lr`.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/losses/confidence.py`](../../src/neural_data_decoding/training/losses/confidence.py) — `_compute_confidence_stream_loss`, the blend + Eq. 10.
- [06.4 EMA prior normalization](06.4_the_ema_prior_normalization_deep_dive.ipynb) — the same running-estimate pattern for the loss priors.
- [02.5 autograd basics](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb) — why detaching the history changes what the gradient sees.

Next up: **[06.10 NaN-masked reconstruction](06.10_nan_masked_reconstruction.ipynb)** — the two-layer NaN handling in the reconstruction loss.